<a href="https://colab.research.google.com/github/Bunseki2/NeurIPS-2024---Predict-New-Medicines-with-BELKA/blob/main/solution_4thplace_comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is based on the copy from @Ah(https://www.kaggle.com/code/ahmedelfazouan/belka-1dcnn-starter-with-all-data) and modified using PCA.

In [ ]:
!pip install fastparquet -q


[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip


In [ ]:
import gc
import os
import pickle
import random
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score as APS

In [ ]:
class CFG:

    PREPROCESS = False
    EPOCHS = 20
    BATCH_SIZE = 4096
    LR = 1e-3
    WD = 0.05

    NBR_FOLDS = 15
    SELECTED_FOLDS = [0]

    SEED = 2024

In [ ]:
import tensorflow as tf

# Detect hardware, return appropriate distribution strategy
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu="local") # "local" for 1VM TPU
    strategy = tf.distribute.TPUStrategy(tpu)
    print("Running on TPU")
    print("REPLICAS: ", strategy.num_replicas_in_sync)
except tf.errors.NotFoundError:
    print("Not on TPU")

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local
INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU

# Preprocessing

In [ ]:
# Check if preprocessing is enabled in the configuration

if CFG.PREPROCESS:
    # Define a dictionary for encoding different characters in the SMILES strings
    enc = {'l': 1, 'y': 2, '@': 3, '3': 4, 'H': 5, 'S': 6, 'F': 7, 'C': 8, 'r': 9, 's': 10, '/': 11, 'c': 12, 'o': 13,
           '+': 14, 'I': 15, '5': 16, '(': 17, '2': 18, ')': 19, '9': 20, 'i': 21, '#': 22, '6': 23, '8': 24, '4': 25, '=': 26,
           '1': 27, 'O': 28, '[': 29, 'D': 30, 'B': 31, ']': 32, 'N': 33, '7': 34, 'n': 35, '-': 36}

    # Read the training data from a parquet file
    train_raw = pd.read_parquet('/kaggle/input/leash-BELKA/train.parquet')

    # Get the SMILES strings for the 'BRD4' protein name
    smiles = train_raw[train_raw['protein_name']=='BRD4']['molecule_smiles'].values

    # Assert that the SMILES strings for 'BRD4', 'HSA', and 'sEH' are identical across the corresponding proteins
    assert (smiles!=train_raw[train_raw['protein_name']=='HSA']['molecule_smiles'].values).sum() == 0
    assert (smiles!=train_raw[train_raw['protein_name']=='sEH']['molecule_smiles'].values).sum() == 0

    # Function to encode a SMILES string using the predefined dictionary and pad to a fixed length
    def encode_smile(smile):
        tmp = [enc[i] for i in smile]
        tmp = tmp + [0]*(142-len(tmp))
        return np.array(tmp).astype(np.uint8)

    # Apply the encoding function to each SMILES string in parallel using 96 CPU cores
    smiles_enc = joblib.Parallel(n_jobs=96)(joblib.delayed(encode_smile)(smile) for smile in tqdm(smiles))

    # Stack the encoded SMILES arrays into a single 2D array
    smiles_enc = np.stack(smiles_enc)

     # Create a DataFrame for the encoded SMILES with column names as 'enc0', 'enc1', ..., 'enc141'
    train = pd.DataFrame(smiles_enc, columns = [f'enc{i}' for i in range(142)])

    # Add the bind labels for the different protein targets ('BRD4', 'HSA', 'sEH')
    train['bind1'] = train_raw[train_raw['protein_name']=='BRD4']['binds'].values
    train['bind2'] = train_raw[train_raw['protein_name']=='HSA']['binds'].values
    train['bind3'] = train_raw[train_raw['protein_name']=='sEH']['binds'].values

    # Save the processed training data to a parquet file
    train.to_parquet('train_enc.parquet')

     # Read the test data from a parquet file
    test_raw = pd.read_parquet('/kaggle/input/leash-BELKA/test.parquet')

    # Get the SMILES strings from the test data
    smiles = test_raw['molecule_smiles'].values

    # Apply the encoding function to each SMILES string in parallel for the test data
    smiles_enc = joblib.Parallel(n_jobs=96)(joblib.delayed(encode_smile)(smile) for smile in tqdm(smiles))

    # Stack the encoded SMILES arrays into a single 2D array
    smiles_enc = np.stack(smiles_enc)

    # Create a DataFrame for the encoded SMILES for the test data with columns 'enc0', 'enc1', ..., 'enc141'
    test = pd.DataFrame(smiles_enc, columns = [f'enc{i}' for i in range(142)])

    # Save the processed test data to a parquet file
    test.to_parquet('test_enc.parquet')

# If preprocessing is not enabled, load pre-encoded training and test data
else:
    # Load the pre-encoded training data and test data from a parquet file
    train = pd.read_parquet('/kaggle/input/belka-enc-dataset/train_enc.parquet')
    test = pd.read_parquet('/kaggle/input/belka-enc-dataset/test_enc.parquet')

In [ ]:
train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
import numpy as np

In [ ]:
NUM_FEATURE_COLUMNS = [i for i in train.columns if i not in ["bind1","bind2","bind3"]]

In [ ]:
# Create a pipeline that first scales the data and then applies PCA
pipeline = Pipeline([('scaling', StandardScaler()), ('pca', PCA(n_components=3))])

# Initialize a PCA object directly with 3 components
pca = PCA(n_components=3)
# Fit the pipeline to the numerical features of the training data and transform it
# This applies both scaling and PCA to the specified columns
pca_result = pipeline.fit_transform(train[NUM_FEATURE_COLUMNS].values)

# Add new columns to the training dataframe to store the principal component values
# The results from PCA are accessed by their column index (0, 1, and 2)
# The data type is cast to 'float16' to potentially reduce memory usage
train['pca-one'] = pca_result[:,0].astype('float16')
train['pca-two'] = pca_result[:,1].astype('float16')
train['pca-three'] = pca_result[:,2].astype('float16')


In [ ]:
#train.to_parquet('/kaggle/working/train_desc_pca.parquet')

In [ ]:
# Apply the *already fitted* pipeline to the numerical features of the test data to transform it.
# It's crucial to use the 'transform' method here, not 'fit_transform', to apply the same scaling and PCA learned from the training data.
pca_result = pipeline.transform(test[NUM_FEATURE_COLUMNS].values)

# Add new columns to the test dataframe to store the principal component values obtained from the transformed test data.
# These new PCA features in the test set are based on the transformations learned from the training set.
test['pca-one'] = pca_result[:,0].astype('float16')
test['pca-two'] = pca_result[:,1].astype('float16')
test['pca-three'] = pca_result[:,2].astype('float16')

In [ ]:
#test.to_parquet('/kaggle/working/test_desc_pca.parquet')

# Modeling

In [ ]:
def my_model():
    # Define model within the distribution strategy scope (for multi-GPU or TPU training)
    with strategy.scope():
        INP_LEN = 145             # Input sequence length
        NUM_FILTERS = 32          # Base number of filters for the convolutional layers
        hidden_dim = 128          # Dimension of the embedding space

        # Input layer expecting a sequence of integer-encoded tokens of fixed length INP_LEN
        inputs = tf.keras.layers.Input(shape=(INP_LEN,)) #, dtype='int32')

        # Embedding layer to map integer tokens to dense vectors
        # input_dim=36: vocabulary size
        # output_dim=hidden_dim: embedding vector size
        # mask_zero=True: ignore padding tokens (token id 0)
        x = tf.keras.layers.Embedding(input_dim=36, output_dim=hidden_dim, input_length=INP_LEN, mask_zero = True)(inputs)

        # First convolutional layer to extract local features from embedded sequences
        x = tf.keras.layers.Conv1D(filters=NUM_FILTERS, kernel_size=7,  activation='relu', padding='valid',  strides=1)(x)
        # Second convolutional layer with increased filters
        x = tf.keras.layers.Conv1D(filters=NUM_FILTERS*2, kernel_size=7,  activation='relu', padding='valid',  strides=1)(x)
        # Third convolutional layer with even more filters to extract higher-level feature
        x = tf.keras.layers.Conv1D(filters=NUM_FILTERS*3, kernel_size=7,  activation='relu', padding='valid',  strides=1)(x)
        # Global max pooling layer to reduce the feature map to a fixed-size vector
        x = tf.keras.layers.GlobalMaxPooling1D()(x)

        # Fully connected dense layers with ReLU activation and dropout regularization
        x = tf.keras.layers.Dense(1024, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.1)(x)
        x = tf.keras.layers.Dense(1024, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.1)(x)
        x = tf.keras.layers.Dense(512, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.1)(x)

        # Output layer with 3 sigmoid units (e.g., for multi-label classification with 3 classes)
        outputs = tf.keras.layers.Dense(3, activation='sigmoid')(x)

        # Define the Keras model
        model = tf.keras.models.Model(inputs = inputs, outputs = outputs)

        # Define the optimizer with learning rate and weight decay from config
        optimizer = tf.keras.optimizers.Adam(learning_rate=CFG.LR, weight_decay = CFG.WD)

        # Binary crossentropy loss for multi-label classification
        loss = 'binary_crossentropy'

        # Evaluation metric: Average Precision (area under the Precision-Recall curve)
        weighted_metrics = [tf.keras.metrics.AUC(curve='PR', name = 'avg_precision')]

        # Compile the model with specified loss, optimizer, and metric
        model.compile(
        loss=loss,
        optimizer=optimizer,
        weighted_metrics=weighted_metrics,
        )
        return model

# Train & Inference

In [ ]:
# Define features by excluding the target columns from the dataset
FEATURES = [i for i in train.columns if i not in ["bind1","bind2","bind3"]]

# Define target columns (multi-label classification problem)
TARGETS = ['bind1', 'bind2', 'bind3']

# Stratified K-Fold cross-validation based on the sum of labels per sample (to preserve label distribution across folds)
skf = StratifiedKFold(n_splits = CFG.NBR_FOLDS, shuffle = True, random_state = 42)

all_preds = []  # Store predictions from each fold

# Loop through each fold generated by StratifiedKFold
for fold,(train_idx, valid_idx) in enumerate(skf.split(train, train[TARGETS].sum(1))):

    # Skip folds not selected for training (controlled via config)
    if fold not in CFG.SELECTED_FOLDS:
        continue;

    # Create training and validation sets for this fold
    X_train = train.loc[train_idx, FEATURES]
    y_train = train.loc[train_idx, TARGETS]
    X_val = train.loc[valid_idx, FEATURES]
    y_val = train.loc[valid_idx, TARGETS]

    #xgb_model = XGBClassifier(use_label_encoder=False, n_estimators=500, random_state=43, )
    #xgb_model.fit(X_train, y_train)

    # Define Keras callbacks for training
    es = tf.keras.callbacks.EarlyStopping(patience=5, monitor="val_loss", mode='min', verbose=1) # Stop training early if validation loss doesn't improve

    checkpoint = tf.keras.callbacks.ModelCheckpoint(monitor='val_loss', filepath=f"model-{fold}.h5",
                                                        save_best_only=True, save_weights_only=True,
                                                    mode='min') # Save model weights with best validation loss

    reduce_lr_loss = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.05, patience=5, verbose=1) # Reduce learning rate if validation loss plateaus

    # Initialize a new instance of the model
    model = my_model()

    # Train the model on current fold's training data, with validation
    history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=CFG.EPOCHS,
            callbacks=[checkpoint, reduce_lr_loss, es],
            batch_size=CFG.BATCH_SIZE,
            verbose=1,
        )

    # Load the best model weights saved during training
    model.load_weights(f"model-{fold}.h5")

    # Predict on validation set and evaluate
    oof = model.predict(X_val, batch_size = 2*CFG.BATCH_SIZE)
    print('fold :', fold, 'CV score =', APS(y_val, oof, average = 'micro'))

    # Save the trained model for this fold
    model.save(f"/kaggle/working/my_model-{fold}.keras")

    # Predict on test data using the current fold's model
    preds = model.predict(test, batch_size = 2*CFG.BATCH_SIZE)
    all_preds.append(preds)

# Average predictions from all folds to get final ensemble result
preds = np.mean(all_preds, 0)

# Submission

In [ ]:
tst = pd.read_parquet('/kaggle/input/test.parquet')
tst['binds'] = 0
tst.loc[tst['protein_name']=='BRD4', 'binds'] = preds[(tst['protein_name']=='BRD4').values, 0]
tst.loc[tst['protein_name']=='HSA', 'binds'] = preds[(tst['protein_name']=='HSA').values, 1]
tst.loc[tst['protein_name']=='sEH', 'binds'] = preds[(tst['protein_name']=='sEH').values, 2]
tst[['id', 'binds']].to_csv('submission.csv', index = False)

## License
This Notebook has been released under the MIT open source license.

